In [2]:

import numpy as np
import random

# ==========================================
# 1. MANUFACTURING JOB-SHOP INPUT DATA
# ==========================================

# Format: (Machine, Processing Time)

jobs = {
    0: [('M1', 3), ('M2', 2), ('M3', 2)],
    1: [('M2', 2), ('M1', 4), ('M3', 3)],
    2: [('M3', 4), ('M2', 3), ('M1', 2)],
    3: [('M1', 2), ('M3', 3), ('M2', 4)]
}

n_jobs = len(jobs)
n_machines = 3

# ==========================================
# 2. ACO PARAMETERS
# ==========================================

n_ants = 20
iterations = 100

alpha = 1.0       # Pheromone importance
beta = 2.0        # Heuristic importance
rho = 0.5         # Evaporation rate
Q = 100           # Pheromone deposit constant

# Assign unique ID to every operation
operation_ids = {}
operation_details = {}

op_id = 0

for job in jobs:

    operation_ids[job] = []

    for index, (machine, duration) in enumerate(jobs[job]):

        operation_ids[job].append(op_id)

        operation_details[op_id] = (
            job, index, machine, duration
        )

        op_id += 1

n_operations = op_id

# Initial pheromone matrix
pheromone = np.ones(
    (n_operations + 1, n_operations)
)

# ==========================================
# 3. SCHEDULE GENERATION
# ==========================================

def generate_schedule():

    job_next = [0] * n_jobs

    job_available = [0] * n_jobs

    machine_available = {
        'M1': 0,
        'M2': 0,
        'M3': 0
    }

    schedule = []
    sequence = []

    current_operation = None

    # Schedule all operations
    while len(schedule) < n_operations:

        # Find operations whose predecessors
        # have already been completed

        available_operations = []

        for job in range(n_jobs):

            index = job_next[job]

            if index < len(jobs[job]):

                machine, duration = jobs[job][index]

                operation_id = operation_ids[job][index]

                available_operations.append(
                    (operation_id, job, index,
                     machine, duration)
                )

        # Calculate heuristic information
        probabilities = []

        for operation in available_operations:

            operation_id, job, index, machine, duration = operation

            # Remaining processing time of this job
            remaining_time = sum(
                t for m, t in jobs[job][index:]
            )

            # Heuristic information
            eta = 1 / remaining_time

            # Pheromone information
            tau = pheromone[
                0 if current_operation is None
                else current_operation,
                operation_id
            ]

            value = (tau ** alpha) * (eta ** beta)

            probabilities.append(value)

        # Convert values into probabilities
        probabilities = np.array(probabilities)

        probabilities = probabilities / np.sum(probabilities)

        # Select next operation using roulette wheel
        selected_index = np.random.choice(
            len(available_operations),
            p=probabilities
        )

        operation = available_operations[selected_index]

        operation_id, job, index, machine, duration = operation

        # Calculate operation start and end times
        start_time = max(
            job_available[job],
            machine_available[machine]
        )

        end_time = start_time + duration

        # Update availability
        job_available[job] = end_time

        machine_available[machine] = end_time

        # Store operation details
        schedule.append({
            'job': f'J{job + 1}',
            'operation': index + 1,
            'machine': machine,
            'start': start_time,
            'end': end_time
        })

        sequence.append(operation_id)

        # Move to next operation of this job
        job_next[job] += 1

        current_operation = operation_id

    # Calculate makespan
    makespan = max(
        item['end'] for item in schedule
    )

    return sequence, schedule, makespan


# ==========================================
# 4. ACO OPTIMIZATION
# ==========================================

best_sequence = None
best_schedule = None
best_makespan = float('inf')

for iteration in range(iterations):

    ant_solutions = []

    # Generate solutions using ants
    for ant in range(n_ants):

        sequence, schedule, makespan = generate_schedule()

        ant_solutions.append(
            (sequence, schedule, makespan)
        )

        # Update global best
        if makespan < best_makespan:

            best_makespan = makespan
            best_sequence = sequence.copy()
            best_schedule = schedule.copy()

    # ======================================
    # 5. PHEROMONE EVAPORATION
    # ======================================

    pheromone *= (1 - rho)

    # ======================================
    # 6. PHEROMONE DEPOSITION
    # ======================================

    for sequence, schedule, makespan in ant_solutions:

        deposit = Q / makespan

        previous = 0

        for operation in sequence:

            pheromone[previous, operation] += deposit

            previous = operation


# ==========================================
# 7. DISPLAY OPTIMIZED RESULTS
# ==========================================

print("\n========================================")
print(" ACO JOB-SHOP SCHEDULING OPTIMIZATION")
print("========================================")

print("\nOPTIMIZED JOB SEQUENCE:")

for operation in best_schedule:

    print(
        operation['job'],
        "->",
        operation['machine'],
        "->",
        f"{operation['start']} - {operation['end']}"
    )

print("\n========================================")
print(" MACHINE-WISE SCHEDULE")
print("========================================")

for machine in ['M1', 'M2', 'M3']:

    print(f"\n{machine}:")

    machine_operations = [
        item for item in best_schedule
        if item['machine'] == machine
    ]

    machine_operations.sort(
        key=lambda x: x['start']
    )

    for item in machine_operations:

        print(
            f"{item['job']} | "
            f"Start: {item['start']} | "
            f"End: {item['end']}"
        )

print("\n========================================")
print(" FINAL OPTIMIZATION RESULT")
print("========================================")

print("Minimum Makespan:", best_makespan)

print("Total Jobs:", n_jobs)

print("Total Machines:", n_machines)

print("Total Operations:", n_operations)

print("========================================")



 ACO JOB-SHOP SCHEDULING OPTIMIZATION

OPTIMIZED JOB SEQUENCE:
J4 -> M1 -> 0 - 2
J2 -> M2 -> 0 - 2
J3 -> M3 -> 0 - 4
J1 -> M1 -> 2 - 5
J3 -> M2 -> 4 - 7
J3 -> M1 -> 7 - 9
J1 -> M2 -> 7 - 9
J1 -> M3 -> 9 - 11
J2 -> M1 -> 9 - 13
J4 -> M3 -> 11 - 14
J2 -> M3 -> 14 - 17
J4 -> M2 -> 14 - 18

 MACHINE-WISE SCHEDULE

M1:
J4 | Start: 0 | End: 2
J1 | Start: 2 | End: 5
J3 | Start: 7 | End: 9
J2 | Start: 9 | End: 13

M2:
J2 | Start: 0 | End: 2
J3 | Start: 4 | End: 7
J1 | Start: 7 | End: 9
J4 | Start: 14 | End: 18

M3:
J3 | Start: 0 | End: 4
J1 | Start: 9 | End: 11
J4 | Start: 11 | End: 14
J2 | Start: 14 | End: 17

 FINAL OPTIMIZATION RESULT
Minimum Makespan: 18
Total Jobs: 4
Total Machines: 3
Total Operations: 12
